# STIR-Net V1 — 24 Spatial-Proposal Causal Evaluation

This notebook evaluates the newly implemented **high-resolution learned spatial-proposal architecture** on the same difficult BlastoSPIM all-cell scene used in the earlier STIR-Net debugging notebooks.

The principal debugging target is:

```text
SOURCE_ID = 9
```

where one current connected component overlaps **9 GT cells**.

## Causal question

> With temporal reasoning completely removed, can the new spatial pathway learn approximately one distinct spatial hypothesis for each real cell inside source component 9?

The notebook deliberately separates the causal chain:

```text
current-frame geometry
        ↓
learned proposal score
        ↓
spatial proposal anchors
        ↓
proposal-local identities
        ↓
query decoder
        ↓
coarse instance masks
```

It does **not** run a long staged overfit.

## Experiment phases

### Phase A — zero-training A/B
Load the exact Notebook-12 step-30 spatial checkpoint and compare:

- `legacy` query construction
- `spatial_proposals` query construction

with the same migrated checkpoint and **no temporal memory**.

### Phase B — proposal/spatial micro-overfit
Train only:

- acquisition/spatial encoder/decoder
- dense heads
- spatial proposal generator

using only the dense spatial objectives:

- foreground
- center heatmap
- global boundary
- internal cell-cell boundary
- proposal-center score

No Hungarian query loss is used here.

Primary gate:

```text
source-9 GT proposal recall @ 1 dref >= 8/9
```

### Phase C — symmetry check
If proposals are recovered, measure whether source-9 hypotheses now have:

- distinct initial anchors
- less-collapsed initial query embeddings
- less-collapsed coarse masks

### Phase D — short spatial-only query bootstrap
Only if Phase B passes the proposal gate, train:

- spatial
- dense
- proposal
- query builder
- query decoder

with temporal state empty and native losses disabled.

The result localizes the next bottleneck:

```text
proposal recall poor
    → spatial representation/proposal mechanism

proposal recall good but masks collapse
    → query decoder / mask representation

proposals + masks separate
    → architecture fixed upstream; move downstream
```


In [ ]:
from pathlib import Path
import copy
import gc
import json
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from learned.stirnet import RefinementCriterion, StirNet
from learned.stirnet.debugging.acceptance.first_overfit import (
    _reduced_config,
    _repo_root,
    build_real_batch,
)
from learned.stirnet.debugging.probes.matching import (
    coarse_dice_for_matches,
    run_matching_probe,
)
from learned.stirnet.model.matcher import target_ids
from learned.stirnet.model.query_builder import (
    QUERY_PRIMARY,
    QUERY_SPLIT,
    QUERY_SPATIAL_PROPOSAL,
)
from learned.stirnet.model.types import (
    StirNetOutput,
    TemporalState,
)
from learned.stirnet.training.checkpoint import load_checkpoint
from learned.stirnet.training.curriculum import model_parameter_groups
from learned.stirnet.training.trainer import move_batch_to_device

# -------------------------------------------------------------------------
# Experiment configuration
# -------------------------------------------------------------------------

SEED = 40266
SOURCE_ID = 9

AMP_DTYPE = torch.float16
BASE_LR = 2e-4

# Short causal screens only.
PROPOSAL_TRAIN_STEPS = 20
PROPOSAL_EVAL_STEPS = {0, 1, 3, 5, 10, 20}

QUERY_TRAIN_STEPS = 20
QUERY_EVAL_STEPS = {0, 5, 10, 20}

# Run query bootstrap only when the proposal mechanism first proves itself.
RUN_QUERY_BOOTSTRAP_IF_GATE_PASSES = True

# Architecture gate: source-9 has nine GT cells.
PROPOSAL_GATE_RECALL_1DREF = 8 / 9

REPO_ROOT = _repo_root(Path.cwd())

DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)

STEP30_CHECKPOINT = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "12_staged_same_sample"
    / "checkpoint_spatial_dense.pt"
)

RUN_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "targeted"
    / "24_spatial_proposal_causal_evaluation"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 24 requires CUDA.")

if not STEP30_CHECKPOINT.exists():
    raise FileNotFoundError(
        "Exact Notebook-12 step-30 checkpoint not found:\n"
        f"{STEP30_CHECKPOINT}"
    )

device = torch.device("cuda")

print("Repository :", REPO_ROOT)
print("Data       :", DATA_DIR)
print("Checkpoint :", STEP30_CHECKPOINT)
print("Run dir    :", RUN_DIR)
print("GPU        :", torch.cuda.get_device_name(0))
print("Proposal micro-steps:", PROPOSAL_TRAIN_STEPS)
print("Query micro-steps   :", QUERY_TRAIN_STEPS)


# 1. Load the same full all-cell scene

This reuses the repository's existing first-overfit scene builder. The full ROI is retained; this is not an isolated source-9 crop.


In [ ]:
batch_cpu, sample_info = build_real_batch(DATA_DIR)

# Keep large GT target maps on CPU, as the repository trainer does.
b = move_batch_to_device(batch_cpu, device)
b["spatial_inputs"] = b["spatial_inputs"].to(dtype=AMP_DTYPE)
b["instance_labels"] = b["instance_labels"].to(dtype=torch.int32)

targets = batch_cpu["targets"]
target = targets[0]

current_labels_native = (
    batch_cpu["instance_labels"][0]
    .detach()
    .cpu()
    .numpy()
    .astype(np.int32, copy=False)
)

gt_labels_native = (
    torch.as_tensor(target["label_map"])
    .detach()
    .cpu()
    .numpy()
    .astype(np.int32, copy=False)
)

spacing_native = (
    batch_cpu["spacing_um"][0]
    .detach()
    .cpu()
    .numpy()
    .astype(np.float64)
)

dref_um = float(batch_cpu["dref_um"][0])

gt_ids = target_ids(target).detach().cpu().long()
gt_centers_cellscale = torch.as_tensor(
    target["centers_cellscale"],
    dtype=torch.float32,
)

source9_gt_ids = np.unique(
    gt_labels_native[current_labels_native == SOURCE_ID]
)
source9_gt_ids = source9_gt_ids[source9_gt_ids > 0].astype(int)

source9_gt_id_set = set(source9_gt_ids.tolist())
source9_gt_indices = torch.tensor(
    [
        idx
        for idx, gt_id in enumerate(gt_ids.tolist())
        if int(gt_id) in source9_gt_id_set
    ],
    dtype=torch.long,
)

source9_gt_centers = gt_centers_cellscale[source9_gt_indices].float()

print(json.dumps(sample_info, indent=2, default=float))
print()
print("Source-9 GT IDs   :", source9_gt_ids.tolist())
print("Source-9 GT count :", len(source9_gt_ids))
print("dref_um           :", dref_um)
print("spacing z/y/x     :", spacing_native.tolist())

if len(source9_gt_ids) != 9:
    raise RuntimeError(
        f"Expected source 9 to overlap 9 GT cells; got {len(source9_gt_ids)}."
    )


# 2. Exact spatial-only forward helpers

The repository's normal `StirNet.forward()` always builds temporal state before `bypass_coreasoning` is applied.

For this causal experiment we want a stricter condition:

```text
no CR1
no CR2
no temporal query memory
no temporal queries
```

So the notebook calls the **actual repository modules** directly while supplying an empty `TemporalState` to the existing query builder/decoder.

No model layer is reimplemented.


In [ ]:
def make_empty_temporal(model, *, dtype):
    d_model = int(model.cfg.temporal.d_model)
    return TemporalState(
        tokens=torch.empty((0, d_model), device=device, dtype=dtype),
        ref_um=torch.empty((0, 3), device=device, dtype=torch.float32),
        ref_cellscale=torch.empty((0, 3), device=device, dtype=torch.float32),
        salience=torch.empty((0, 1), device=device, dtype=dtype),
        reliability=torch.empty((0, 1), device=device, dtype=dtype),
        status=torch.empty((0,), device=device, dtype=torch.long),
        edge_index=torch.empty((2, 0), device=device, dtype=torch.long),
        edge_attr=torch.empty((0, 22), device=device, dtype=torch.float32),
        batch_index=torch.empty((0,), device=device, dtype=torch.long),
    )


def load_step30_model(query_mode: str):
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    cfg = _reduced_config()
    cfg.proposals.enabled = True
    cfg.proposals.query_mode = query_mode

    model = StirNet(cfg).to(device)

    info = load_checkpoint(
        STEP30_CHECKPOINT,
        model,
        optimizer=None,
        scheduler=None,
        scaler=None,
        map_location="cpu",
        strict=True,
        migrate_history=True,
    )

    if int(info.get("step", -1)) != 30:
        raise RuntimeError(
            f"Expected checkpoint payload step 30; got {info.get('step')}."
        )

    return model, cfg, info


def forward_spatial_proposal_backbone(model):
    """Spatial+dense+proposal path only. No temporal/query/native work."""
    acq = model.acquisition(
        b["spacing_um"],
        b["dref_um"],
    )

    pyramid = model.encoder(
        b["spatial_inputs"],
        b["spacing_um"],
        acq,
        b.get("spatial_padding_mask"),
    )

    # Explicit spatial-only path: no CR1/CR2.
    e3 = pyramid.features[3]

    e2 = model.decoder.decode_to_e2(
        e3,
        pyramid,
        acq,
    )

    d1, d0, mask_features = model.decoder.decode_from_e2(
        e2,
        pyramid,
        acq,
    )

    dense = model.dense_heads(d0)

    proposal_state, proposal_score_logits = (
        model.spatial_proposal_generator(
            d0,
            e2,
            b["spatial_inputs"],
            dense,
            b["instance_labels"],
            b["spacing_um"],
            pyramid.spacings_um[2],
            b["dref_um"],
            b["instance_ids"],
            b["instance_batch"],
            b["instance_centroids_um"],
            b.get("spatial_padding_mask"),
        )
    )

    dense = dict(dense)
    dense["proposal_score_logits"] = proposal_score_logits

    return {
        "acq": acq,
        "pyramid": pyramid,
        "e3": e3,
        "e2": e2,
        "d1": d1,
        "d0": d0,
        "mask_features": mask_features,
        "dense": dense,
        "proposals": proposal_state,
    }


def forward_spatial_only_full(model, *, query_mode: str):
    """
    Full spatial -> proposal/query -> coarse-mask path with a truly empty
    temporal state. Native embeddings are produced, but native masks are not
    rendered here.
    """
    spatial = forward_spatial_proposal_backbone(model)

    e3 = spatial["e3"]
    e2 = spatial["e2"]
    d1 = spatial["d1"]
    pyramid = spatial["pyramid"]
    dense = spatial["dense"]
    proposal_state = spatial["proposals"]

    if query_mode == "legacy":
        proposal_for_queries = None
    elif query_mode == "spatial_proposals":
        proposal_for_queries = proposal_state
    else:
        raise ValueError(query_mode)

    temporal = make_empty_temporal(
        model,
        dtype=e2.dtype,
    )

    qstate = model.query_builder(
        e2,
        pyramid.spacings_um[2],
        b["instance_labels"],
        b["instance_features"],
        b["instance_ids"],
        b["instance_batch"],
        b["instance_centroids_um"],
        b["dref_um"],
        temporal,
        memory_ablation="full",
        return_debug=False,
        full_attention=False,
        proposal_state=proposal_for_queries,
        query_mode=query_mode,
    )

    initial_references = qstate.references_cellscale.clone()
    initial_embeddings = qstate.embeddings.clone()

    qstate, decoder_outputs = model.query_decoder(
        qstate,
        [e3, e2, d1],
        [
            pyramid.spacings_um[3],
            pyramid.spacings_um[2],
            pyramid.spacings_um[1],
        ],
        b["instance_labels"],
        b["dref_um"],
        temporal,
        memory_ablation="full",
        return_debug=False,
        full_attention=False,
    )

    final = decoder_outputs[-1]
    native_embeddings = model.native_mask_head(qstate.embeddings)

    debug = {
        "initial_query_embeddings": initial_embeddings,
        "query_layer_references_cellscale": torch.stack(
            [layer["centers_cellscale"] for layer in decoder_outputs],
            dim=0,
        ),
        "proposal_score_logits": dense["proposal_score_logits"],
    }

    return StirNetOutput(
        exist_logits=final["exist_logits"],
        centers_cellscale=final["centers_cellscale"],
        coarse_mask_logits=final["coarse_mask_logits"],
        coarse_spacing_um=final["coarse_spacing_um"],
        query_embeddings=qstate.embeddings,
        native_mask_embeddings=native_embeddings,
        query_types=qstate.query_types,
        query_padding_mask=qstate.padding_mask,
        source_instance_ids=qstate.source_instance_ids,
        query_initial_references_cellscale=initial_references,
        temporal_salience=qstate.temporal_salience,
        temporal_reliability=qstate.temporal_reliability,
        aux_outputs=decoder_outputs[:-1],
        dense_outputs=dense,
        mask_features=spatial["mask_features"],
        spacing_um=b["spacing_um"],
        dref_um=b["dref_um"],
        instance_labels=b["instance_labels"],
        debug=debug,
        proposals=proposal_state,
    )


# 3. Proposal and query diagnostics

All proposal-center distances below are already in **cell-scale units**, i.e. normalized by `dref_um`.

Therefore:

```text
distance = 0.5  →  0.5 dref
distance = 1.0  →  1.0 dref
```


In [ ]:
def upper_triangle_values(matrix: torch.Tensor):
    if matrix.shape[0] < 2:
        return torch.empty((0,), dtype=matrix.dtype)
    mask = torch.triu(
        torch.ones_like(matrix, dtype=torch.bool),
        diagonal=1,
    )
    return matrix[mask]


def cosine_pair_metrics(embeddings: torch.Tensor):
    if embeddings.shape[0] < 2:
        return {
            "cos_mean": np.nan,
            "cos_median": np.nan,
            "cos_max": np.nan,
        }

    normalized = F.normalize(
        embeddings.float(),
        dim=-1,
    )
    pair = upper_triangle_values(
        normalized @ normalized.T
    )

    return {
        "cos_mean": float(pair.mean()),
        "cos_median": float(pair.median()),
        "cos_max": float(pair.max()),
    }


def reference_pair_metrics(references: torch.Tensor):
    if references.shape[0] < 2:
        return {
            "pair_dist_mean_dref": np.nan,
            "pair_dist_median_dref": np.nan,
            "pair_dist_min_dref": np.nan,
        }

    pair = torch.pdist(
        references.float(),
        p=2,
    )

    return {
        "pair_dist_mean_dref": float(pair.mean()),
        "pair_dist_median_dref": float(pair.median()),
        "pair_dist_min_dref": float(pair.min()),
    }


def proposal_recall(references: torch.Tensor, radius_dref: float):
    if references.shape[0] == 0:
        return 0.0, torch.full(
            (len(source9_gt_centers),),
            float("inf"),
        )

    distances = torch.cdist(
        source9_gt_centers.float(),
        references.float().cpu(),
    )
    nearest = distances.min(dim=1).values
    recall = float(
        (nearest <= float(radius_dref))
        .float()
        .mean()
    )
    return recall, nearest


def proposal_metrics(proposals):
    valid = ~proposals.padding_mask[0].detach().cpu()
    refs = proposals.references_cellscale[0].detach().float().cpu()[valid]
    scores = proposals.scores[0].detach().float().cpu()[valid]
    sources = proposals.source_instance_ids[0].detach().cpu()[valid]
    fallback = proposals.fallback_mask[0].detach().cpu()[valid]
    embeddings = proposals.embeddings[0].detach().float().cpu()[valid]

    learned = ~fallback
    source9 = sources == SOURCE_ID
    source9_learned = source9 & learned

    learned_refs = refs[learned]
    s9_learned_refs = refs[source9_learned]

    recall_05_all, nearest_05_source = proposal_recall(
        learned_refs,
        0.5,
    )
    recall_10_all, nearest_10_source = proposal_recall(
        learned_refs,
        1.0,
    )

    recall_05_s9, nearest_05_s9 = proposal_recall(
        s9_learned_refs,
        0.5,
    )
    recall_10_s9, nearest_10_s9 = proposal_recall(
        s9_learned_refs,
        1.0,
    )

    emb_metrics = cosine_pair_metrics(
        embeddings[source9_learned]
    )
    ref_metrics = reference_pair_metrics(
        refs[source9_learned]
    )

    row = {
        "proposal_total": int(valid.sum()),
        "proposal_learned": int(learned.sum()),
        "proposal_fallback": int(fallback.sum()),
        "source9_total": int(source9.sum()),
        "source9_learned": int(source9_learned.sum()),
        "source9_fallback": int((source9 & fallback).sum()),
        "source9_recall_0p5_all_learned": recall_05_all,
        "source9_recall_1p0_all_learned": recall_10_all,
        "source9_recall_0p5_source9_learned": recall_05_s9,
        "source9_recall_1p0_source9_learned": recall_10_s9,
        "source9_nearest_mean_dref_all_learned": float(
            nearest_10_source.mean()
        ),
        "source9_nearest_max_dref_all_learned": float(
            nearest_10_source.max()
        ),
        "source9_score_mean": (
            float(scores[source9].mean())
            if bool(source9.any())
            else np.nan
        ),
        **{
            f"source9_embedding_{k}": v
            for k, v in emb_metrics.items()
        },
        **{
            f"source9_reference_{k}": v
            for k, v in ref_metrics.items()
        },
    }

    artifacts = {
        "refs": refs,
        "scores": scores,
        "sources": sources,
        "fallback": fallback,
        "embeddings": embeddings,
        "source9_gt_nearest_dref_all_learned": nearest_10_source,
        "source9_gt_nearest_dref_source9_learned": nearest_10_s9,
    }

    return row, artifacts


def pairwise_soft_dice(mask_logits: torch.Tensor):
    if mask_logits.shape[0] < 2:
        return {
            "mask_pair_dice_mean": np.nan,
            "mask_pair_dice_median": np.nan,
            "mask_pair_dice_max": np.nan,
        }

    probs = mask_logits.float().sigmoid().flatten(1)
    intersection = probs @ probs.T
    sums = probs.sum(dim=1)
    dice = (
        2 * intersection + 1e-6
    ) / (
        sums[:, None] + sums[None, :] + 1e-6
    )
    pair = upper_triangle_values(dice)

    return {
        "mask_pair_dice_mean": float(pair.mean()),
        "mask_pair_dice_median": float(pair.median()),
        "mask_pair_dice_max": float(pair.max()),
    }


def query_metrics(outputs, cfg, *, query_mode: str):
    valid = ~outputs.query_padding_mask[0].detach().cpu()
    qtypes = outputs.query_types[0].detach().cpu()
    sources = outputs.source_instance_ids[0].detach().cpu()

    if query_mode == "legacy":
        source9_slots = (
            valid
            & (sources == SOURCE_ID)
            & (
                (qtypes == QUERY_PRIMARY)
                | (qtypes == QUERY_SPLIT)
            )
        )
        allowed_types = {
            QUERY_PRIMARY,
            QUERY_SPLIT,
        }
    else:
        source9_slots = (
            valid
            & (sources == SOURCE_ID)
            & (qtypes == QUERY_SPATIAL_PROPOSAL)
        )
        allowed_types = {
            QUERY_SPATIAL_PROPOSAL,
        }

    source9_indices = torch.nonzero(
        source9_slots,
        as_tuple=False,
    ).flatten()

    initial_refs = (
        outputs.query_initial_references_cellscale[0]
        .detach()
        .float()
        .cpu()[source9_indices]
    )

    final_refs = (
        outputs.centers_cellscale[0]
        .detach()
        .float()
        .cpu()[source9_indices]
    )

    initial_embeddings = (
        outputs.debug["initial_query_embeddings"][0]
        .detach()
        .float()
        .cpu()[source9_indices]
    )

    final_embeddings = (
        outputs.query_embeddings[0]
        .detach()
        .float()
        .cpu()[source9_indices]
    )

    source9_logits = (
        outputs.coarse_mask_logits[0]
        .detach()
        .float()
        .cpu()[source9_indices]
    )

    matching = run_matching_probe(
        outputs,
        targets,
    )
    match = matching.matches[0]

    dice_by_query = coarse_dice_for_matches(
        outputs.coarse_mask_logits[0],
        target,
        match,
    )

    matched_target_ids = []
    matched_dice = []
    matched_queries = []

    for pred_idx, target_idx in zip(
        match.pred_indices.detach().cpu().tolist(),
        match.target_indices.detach().cpu().tolist(),
    ):
        pred_idx = int(pred_idx)
        target_idx = int(target_idx)

        gt_id = int(gt_ids[target_idx])
        qtype = int(qtypes[pred_idx])

        if (
            gt_id in source9_gt_id_set
            and qtype in allowed_types
        ):
            matched_target_ids.append(gt_id)
            matched_queries.append(pred_idx)
            if pred_idx in dice_by_query:
                matched_dice.append(
                    float(dice_by_query[pred_idx])
                )

    result = {
        "query_mode": query_mode,
        "valid_query_count": int(valid.sum()),
        "source9_seed_query_count": int(len(source9_indices)),
        "source9_distinct_gt_matched_by_seed_type": int(
            len(set(matched_target_ids))
        ),
        "source9_matched_query_count": int(
            len(matched_target_ids)
        ),
        "source9_matched_coarse_dice_mean": (
            float(np.mean(matched_dice))
            if matched_dice
            else np.nan
        ),
        "source9_matched_coarse_dice_min": (
            float(np.min(matched_dice))
            if matched_dice
            else np.nan
        ),
        "source9_exist_prob_mean": (
            float(
                outputs.exist_logits[0, source9_indices.to(
                    outputs.exist_logits.device
                )]
                .detach()
                .float()
                .sigmoid()
                .mean()
                .cpu()
            )
            if len(source9_indices)
            else np.nan
        ),
        **{
            f"initial_{k}": v
            for k, v in reference_pair_metrics(
                initial_refs
            ).items()
        },
        **{
            f"final_{k}": v
            for k, v in reference_pair_metrics(
                final_refs
            ).items()
        },
        **{
            f"initial_embedding_{k}": v
            for k, v in cosine_pair_metrics(
                initial_embeddings
            ).items()
        },
        **{
            f"final_embedding_{k}": v
            for k, v in cosine_pair_metrics(
                final_embeddings
            ).items()
        },
        **pairwise_soft_dice(
            source9_logits
        ),
    }

    artifacts = {
        "source9_query_indices": source9_indices,
        "matched_target_ids": matched_target_ids,
        "matched_queries": matched_queries,
        "source9_coarse_mask_logits": source9_logits,
        "matching": matching,
    }

    return result, artifacts


# 4. Dense spatial diagnostics

These are secondary safeguards. The main notebook-24 endpoint is proposal decomposition, but we still record foreground Dice and source-9 internal-boundary behavior to detect obvious collateral damage.


In [ ]:
def auc_from_scores(scores, labels):
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)

    positive = labels == 1
    negative = labels == 0

    if positive.sum() == 0 or negative.sum() == 0:
        return float("nan")

    order = np.argsort(scores, kind="mergesort")
    ranks = np.empty(len(scores), dtype=np.float64)
    ranks[order] = np.arange(1, len(scores) + 1)

    # Correct average ranks for ties.
    sorted_scores = scores[order]
    start = 0
    while start < len(scores):
        end = start + 1
        while (
            end < len(scores)
            and sorted_scores[end] == sorted_scores[start]
        ):
            end += 1
        if end - start > 1:
            avg_rank = 0.5 * (
                (start + 1) + end
            )
            ranks[order[start:end]] = avg_rank
        start = end

    n_pos = int(positive.sum())
    n_neg = int(negative.sum())

    return float(
        (
            ranks[positive].sum()
            - n_pos * (n_pos + 1) / 2
        )
        / (n_pos * n_neg)
    )


GT_FOREGROUND = (
    torch.as_tensor(target["foreground"])
    .detach()
    .cpu()
    .bool()
)

GT_INTERNAL = (
    torch.as_tensor(target["internal_boundary"])
    .detach()
    .cpu()
    > 0.5
)

GT_ALL_BOUNDARY = (
    torch.as_tensor(target["boundary"])
    .detach()
    .cpu()
    > 0.5
)

SOURCE9_NATIVE = torch.from_numpy(
    current_labels_native == SOURCE_ID
)


def dense_diagnostics(dense):
    fg_pred = (
        dense["foreground_logits"][0, 0]
        .detach()
        .float()
        .cpu()
        .sigmoid()
        >= 0.5
    )

    intersection = (
        fg_pred & GT_FOREGROUND
    ).sum().float()

    fg_dice = float(
        (
            2 * intersection + 1e-6
        ) / (
            fg_pred.sum()
            + GT_FOREGROUND.sum()
            + 1e-6
        )
    )

    boundary_prob = (
        dense["boundary_logits"][0, 0]
        .detach()
        .float()
        .cpu()
        .sigmoid()
    )

    positive = SOURCE9_NATIVE & GT_INTERNAL
    negative = (
        SOURCE9_NATIVE
        & GT_FOREGROUND
        & ~GT_ALL_BOUNDARY
    )

    scores = torch.cat(
        [
            boundary_prob[positive],
            boundary_prob[negative],
        ]
    ).numpy()

    labels = np.concatenate(
        [
            np.ones(
                int(positive.sum()),
                dtype=np.int64,
            ),
            np.zeros(
                int(negative.sum()),
                dtype=np.int64,
            ),
        ]
    )

    return {
        "foreground_hard_dice": fg_dice,
        "source9_internal_boundary_auc": auc_from_scores(
            scores,
            labels,
        ),
        "source9_internal_recall_0p5": (
            float(
                (
                    boundary_prob[positive]
                    >= 0.5
                )
                .float()
                .mean()
            )
            if bool(positive.any())
            else np.nan
        ),
    }


# 5. Phase A — zero-training A/B

Both arms start from the exact same step-30 checkpoint.

The legacy arm is evaluated first and then deleted from GPU memory. The spatial-proposal model is then loaded and retained for Phase B.


In [ ]:
@torch.no_grad()
def evaluate_full_snapshot(
    model,
    cfg,
    *,
    query_mode,
    label,
):
    model.eval()

    torch.cuda.reset_peak_memory_stats()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        outputs = forward_spatial_only_full(
            model,
            query_mode=query_mode,
        )

    p_metrics, p_artifacts = proposal_metrics(
        outputs.proposals
    )
    q_metrics, q_artifacts = query_metrics(
        outputs,
        cfg,
        query_mode=query_mode,
    )
    d_metrics = dense_diagnostics(
        outputs.dense_outputs
    )

    row = {
        "label": label,
        **p_metrics,
        **q_metrics,
        **d_metrics,
        "peak_cuda_gib": float(
            torch.cuda.max_memory_allocated()
            / 1024**3
        ),
    }

    # Retain compact CPU artifacts only.
    compact = {
        "proposal": p_artifacts,
        "query": {
            key: value
            for key, value in q_artifacts.items()
            if key != "matching"
        },
        "proposal_score_logits": (
            outputs.dense_outputs[
                "proposal_score_logits"
            ][0, 0]
            .detach()
            .float()
            .cpu()
        ),
    }

    del outputs
    gc.collect()
    torch.cuda.empty_cache()

    return row, compact


legacy_model, legacy_cfg, legacy_load = (
    load_step30_model("legacy")
)

legacy_row, legacy_artifacts = (
    evaluate_full_snapshot(
        legacy_model,
        legacy_cfg,
        query_mode="legacy",
        label="step30_legacy",
    )
)

print("Legacy migration notes:")
for note in legacy_load.get(
    "model_migration",
    [],
):
    if (
        "query_builder.type_embedding"
        in note
        or "spatial_proposal"
        in note
    ):
        print(" ", note)

del legacy_model
gc.collect()
torch.cuda.empty_cache()

proposal_model, proposal_cfg, proposal_load = (
    load_step30_model("spatial_proposals")
)

proposal_step0_row, proposal_step0_artifacts = (
    evaluate_full_snapshot(
        proposal_model,
        proposal_cfg,
        query_mode="spatial_proposals",
        label="step30_spatial_proposals",
    )
)

phase_a_df = pd.DataFrame(
    [
        legacy_row,
        proposal_step0_row,
    ]
)

phase_a_df.to_csv(
    RUN_DIR / "phase_a_zero_training_ab.csv",
    index=False,
)

display(
    phase_a_df[
        [
            "label",
            "proposal_learned",
            "proposal_fallback",
            "source9_learned",
            "source9_fallback",
            "source9_recall_0p5_all_learned",
            "source9_recall_1p0_all_learned",
            "source9_recall_1p0_source9_learned",
            "source9_seed_query_count",
            "initial_pair_dist_mean_dref",
            "initial_embedding_cos_mean",
            "mask_pair_dice_mean",
            "source9_distinct_gt_matched_by_seed_type",
            "source9_matched_coarse_dice_mean",
            "foreground_hard_dice",
            "source9_internal_boundary_auc",
            "peak_cuda_gib",
        ]
    ]
)


## Phase-A interpretation

Do **not** reject the architecture merely because the zero-training proposal recall is still close to the old center-head recall.

The new residual proposal paths and local proposal encoder did not exist in the step-30 checkpoint. The first decisive question is whether **short spatial-only optimization** can now turn current-frame geometry into multiple useful anchors.


In [ ]:
print("Source-9 nearest learned-proposal distance per GT cell at step 0:")
nearest = (
    proposal_step0_artifacts["proposal"][
        "source9_gt_nearest_dref_all_learned"
    ]
)

display(
    pd.DataFrame(
        {
            "gt_id": source9_gt_ids,
            "nearest_learned_proposal_dref": nearest.numpy(),
            "within_0p5_dref": (
                nearest <= 0.5
            ).numpy(),
            "within_1p0_dref": (
                nearest <= 1.0
            ).numpy(),
        }
    )
)


# 6. Visualize the zero-training proposal field

The panels use z-max projections for compact inspection.

- GT source-9 centers are marked separately from proposal anchors.
- Learned proposal anchors and fallback anchors are distinguished.
- The current source-9 component is shown as a binary projection.

This visualization is diagnostic only; all acceptance decisions use the 3-D physical distances above.


In [ ]:
def refs_cellscale_to_voxels(refs_cellscale):
    refs_cellscale = torch.as_tensor(
        refs_cellscale,
        dtype=torch.float32,
    ).cpu().numpy()

    refs_um = refs_cellscale * dref_um
    shape = np.asarray(
        current_labels_native.shape,
        dtype=np.float64,
    )
    patch_center_um = (
        0.5
        * (shape - 1)
        * spacing_native
    )
    return (
        refs_um + patch_center_um[None]
    ) / spacing_native[None]


def plot_proposal_scene(
    artifacts,
    *,
    title,
):
    raw = (
        batch_cpu["spatial_inputs"][0, 0]
        .detach()
        .cpu()
        .numpy()
    )

    score = artifacts[
        "proposal_score_logits"
    ].numpy()

    p = artifacts["proposal"]
    refs = p["refs"]
    fallback = p["fallback"]
    sources = p["sources"]

    learned_s9 = (
        (sources == SOURCE_ID)
        & ~fallback
    )
    fallback_s9 = (
        (sources == SOURCE_ID)
        & fallback
    )

    learned_vox = refs_cellscale_to_voxels(
        refs[learned_s9]
    )
    fallback_vox = refs_cellscale_to_voxels(
        refs[fallback_s9]
    )
    gt_vox = refs_cellscale_to_voxels(
        source9_gt_centers
    )

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(20, 5),
    )

    axes[0].imshow(
        raw.max(axis=0),
        cmap="gray",
    )
    axes[0].set_title("raw z-max")

    axes[1].imshow(
        (
            current_labels_native
            == SOURCE_ID
        ).max(axis=0),
        cmap="gray",
    )
    axes[1].set_title("current source 9")

    axes[2].imshow(
        np.isin(
            gt_labels_native,
            source9_gt_ids,
        ).max(axis=0),
        cmap="gray",
    )
    axes[2].set_title("9 GT cells")

    axes[3].imshow(
        score.max(axis=0),
        cmap="viridis",
    )
    axes[3].set_title("proposal score z-max")

    for ax in axes:
        if len(gt_vox):
            ax.scatter(
                gt_vox[:, 2],
                gt_vox[:, 1],
                marker="+",
                s=90,
                label="GT centers",
            )

    for ax in axes:
        if len(learned_vox):
            ax.scatter(
                learned_vox[:, 2],
                learned_vox[:, 1],
                marker="o",
                facecolors="none",
                s=70,
                label="learned source9",
            )
        if len(fallback_vox):
            ax.scatter(
                fallback_vox[:, 2],
                fallback_vox[:, 1],
                marker="x",
                s=70,
                label="fallback source9",
            )

    axes[-1].legend(
        loc="best",
        fontsize=8,
    )

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_proposal_scene(
    proposal_step0_artifacts,
    title="Notebook 24 — step-30 migrated checkpoint, before proposal training",
)


# 7. Phase B — spatial/proposal micro-overfit

Only these repository parameter groups are trainable:

```text
spatial
dense
proposal
```

The query/temporal/native parameter groups remain frozen.

The loss is exactly the permanent dense spatial objective now implemented in the repository:

```text
0.50 * foreground
1.00 * center_heatmap
0.50 * global_boundary
0.25 * internal_boundary
0.75 * proposal_center
```

No query matching or mask loss contributes during this phase.


In [ ]:
proposal_criterion = RefinementCriterion(
    proposal_cfg.losses,
    proposal_cfg.queries,
    proposal_cfg.training,
    proposal_cfg.proposals,
).to(device)

parameter_groups = model_parameter_groups(
    proposal_model
)

# Freeze everything, then expose only the intended causal pathway.
for parameter in proposal_model.parameters():
    parameter.requires_grad_(False)

for group_name in (
    "spatial",
    "dense",
    "proposal",
):
    for parameter in parameter_groups[group_name]:
        parameter.requires_grad_(True)

proposal_optimizer = torch.optim.AdamW(
    [
        {
            "name": group_name,
            "params": parameter_groups[group_name],
            "lr": BASE_LR,
        }
        for group_name in (
            "spatial",
            "dense",
            "proposal",
        )
    ],
    lr=BASE_LR,
    weight_decay=proposal_cfg.training.weight_decay,
)

proposal_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True,
    init_scale=1024.0,
)


def proposal_spatial_objective(spatial):
    dense = spatial["dense"]

    foreground, center, boundary = (
        proposal_criterion._dense_losses(
            dense,
            targets,
        )
    )

    internal = (
        proposal_criterion._internal_boundary_loss(
            dense["boundary_logits"],
            targets,
        )
    )

    proposal_center = (
        proposal_criterion._stream_dense_loss(
            dense["proposal_score_logits"],
            targets,
            "center_heatmap",
            focal=True,
        )
    )

    weights = proposal_cfg.losses

    total = (
        float(weights.foreground) * foreground
        + float(weights.center_heatmap) * center
        + float(weights.boundary) * boundary
        + float(weights.internal_boundary) * internal
        + float(weights.proposal_center) * proposal_center
    )

    return {
        "loss": total,
        "foreground": foreground,
        "center_heatmap": center,
        "boundary": boundary,
        "internal_boundary": internal,
        "proposal_center": proposal_center,
    }


@torch.no_grad()
def evaluate_proposal_snapshot(
    model,
    *,
    step,
):
    model.eval()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        spatial = (
            forward_spatial_proposal_backbone(
                model
            )
        )

        objective = (
            proposal_spatial_objective(
                spatial
            )
        )

    p_metrics, artifacts = proposal_metrics(
        spatial["proposals"]
    )

    d_metrics = dense_diagnostics(
        spatial["dense"]
    )

    row = {
        "step": int(step),
        **p_metrics,
        **d_metrics,
        **{
            f"loss_{key}": float(
                value.detach().float().cpu()
            )
            for key, value in objective.items()
        },
    }

    compact = {
        "proposal": artifacts,
        "proposal_score_logits": (
            spatial["dense"][
                "proposal_score_logits"
            ][0, 0]
            .detach()
            .float()
            .cpu()
        ),
    }

    del spatial, objective
    gc.collect()
    torch.cuda.empty_cache()

    return row, compact


In [ ]:
phase_b_metric_rows = []
phase_b_train_rows = []
phase_b_artifacts = {}

started = time.perf_counter()

for step in range(
    PROPOSAL_TRAIN_STEPS + 1
):
    if step in PROPOSAL_EVAL_STEPS:
        print(
            f"[Phase B] evaluating step {step} ..."
        )

        row, artifacts = (
            evaluate_proposal_snapshot(
                proposal_model,
                step=step,
            )
        )

        phase_b_metric_rows.append(row)
        phase_b_artifacts[step] = artifacts

        print(
            "  source9 recall @0.5/1.0 dref = "
            f"{row['source9_recall_0p5_all_learned']:.3f} / "
            f"{row['source9_recall_1p0_all_learned']:.3f} | "
            f"source9 learned={row['source9_learned']} "
            f"fallback={row['source9_fallback']}"
        )

    if step == PROPOSAL_TRAIN_STEPS:
        break

    proposal_model.train()
    proposal_criterion.train()

    proposal_optimizer.zero_grad(
        set_to_none=True
    )

    torch.cuda.reset_peak_memory_stats()
    step_started = time.perf_counter()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        spatial = (
            forward_spatial_proposal_backbone(
                proposal_model
            )
        )

        objective = (
            proposal_spatial_objective(
                spatial
            )
        )

    proposal_scaler.scale(
        objective["loss"]
    ).backward()

    proposal_scaler.unscale_(
        proposal_optimizer
    )

    grad_norm = torch.nn.utils.clip_grad_norm_(
        [
            parameter
            for parameter in proposal_model.parameters()
            if parameter.requires_grad
        ],
        float(
            proposal_cfg.training.max_grad_norm
        ),
    )

    proposal_scaler.step(
        proposal_optimizer
    )
    proposal_scaler.update()

    phase_b_train_rows.append(
        {
            "from_step": step,
            "to_step": step + 1,
            **{
                key: float(
                    value.detach().float().cpu()
                )
                for key, value in objective.items()
            },
            "grad_norm_before_clip": float(
                grad_norm.detach().float().cpu()
            ),
            "peak_cuda_gib": float(
                torch.cuda.max_memory_allocated()
                / 1024**3
            ),
            "step_seconds": float(
                time.perf_counter()
                - step_started
            ),
        }
    )

    del spatial, objective
    gc.collect()
    torch.cuda.empty_cache()

phase_b_seconds = (
    time.perf_counter() - started
)

phase_b_df = pd.DataFrame(
    phase_b_metric_rows
)

phase_b_training_df = pd.DataFrame(
    phase_b_train_rows
)

phase_b_df.to_csv(
    RUN_DIR / "phase_b_proposal_metrics.csv",
    index=False,
)

phase_b_training_df.to_csv(
    RUN_DIR / "phase_b_training.csv",
    index=False,
)

print(
    f"Phase B complete in {phase_b_seconds:.1f}s"
)

display(
    phase_b_df[
        [
            "step",
            "source9_learned",
            "source9_fallback",
            "source9_recall_0p5_all_learned",
            "source9_recall_1p0_all_learned",
            "source9_recall_1p0_source9_learned",
            "source9_nearest_mean_dref_all_learned",
            "source9_nearest_max_dref_all_learned",
            "source9_embedding_cos_mean",
            "source9_reference_pair_dist_mean_dref",
            "foreground_hard_dice",
            "source9_internal_boundary_auc",
            "loss_proposal_center",
            "loss_internal_boundary",
            "loss",
        ]
    ]
)


In [ ]:
display(
    phase_b_df[
        [
            "step",
            "source9_learned",
            "source9_fallback",
            "source9_recall_0p5_all_learned",
            "source9_recall_1p0_all_learned",
            "source9_recall_1p0_source9_learned",
            "source9_nearest_mean_dref_all_learned",
            "source9_nearest_max_dref_all_learned",
            "source9_embedding_cos_mean",
            "source9_reference_pair_dist_mean_dref",
            "foreground_hard_dice",
            "source9_internal_boundary_auc",
            "loss_proposal_center",
            "loss_internal_boundary",
            "loss_loss",
        ]
    ]
)

# 8. Phase-B proposal trajectory and gate

The primary gate uses **all learned proposals**, because a useful learned hypothesis is still geometrically valid even if its anchor lands just outside the imperfect current component and receives `source_id=-1`.

The notebook also reports the stricter source-9-associated recall separately.


In [ ]:
fig, ax = plt.subplots(
    figsize=(9, 4)
)

ax.plot(
    phase_b_df["step"],
    phase_b_df[
        "source9_recall_0p5_all_learned"
    ],
    marker="o",
    label="recall @ 0.5 dref",
)

ax.plot(
    phase_b_df["step"],
    phase_b_df[
        "source9_recall_1p0_all_learned"
    ],
    marker="o",
    label="recall @ 1.0 dref",
)

ax.axhline(
    PROPOSAL_GATE_RECALL_1DREF,
    linestyle="--",
    label="8/9 gate",
)

ax.set_xlabel("proposal/spatial micro-training step")
ax.set_ylabel("source-9 GT proposal recall")
ax.set_ylim(0, 1.05)
ax.set_title("Source-9 learned-proposal recovery")
ax.legend()
plt.tight_layout()
plt.show()


phase_b_final = (
    phase_b_df[
        phase_b_df["step"]
        == PROPOSAL_TRAIN_STEPS
    ]
    .iloc[0]
)

proposal_gate_passed = (
    float(
        phase_b_final[
            "source9_recall_1p0_all_learned"
        ]
    )
    >= PROPOSAL_GATE_RECALL_1DREF
)

print("=" * 88)
print("PHASE B — PROPOSAL GATE")
print("=" * 88)
print(
    "Recall @ 0.5 dref:",
    f"{phase_b_final['source9_recall_0p5_all_learned']:.3f}",
)
print(
    "Recall @ 1.0 dref:",
    f"{phase_b_final['source9_recall_1p0_all_learned']:.3f}",
)
print(
    "Source-9 learned/fallback:",
    int(phase_b_final["source9_learned"]),
    "/",
    int(phase_b_final["source9_fallback"]),
)
print(
    "Gate:",
    "PASS" if proposal_gate_passed else "FAIL",
)


In [ ]:
print("Per-GT nearest learned proposal after Phase B:")

final_nearest = (
    phase_b_artifacts[
        PROPOSAL_TRAIN_STEPS
    ]["proposal"][
        "source9_gt_nearest_dref_all_learned"
    ]
)

display(
    pd.DataFrame(
        {
            "gt_id": source9_gt_ids,
            "nearest_learned_proposal_dref": (
                final_nearest.numpy()
            ),
            "within_0p5_dref": (
                final_nearest <= 0.5
            ).numpy(),
            "within_1p0_dref": (
                final_nearest <= 1.0
            ).numpy(),
        }
    )
)

plot_proposal_scene(
    phase_b_artifacts[
        PROPOSAL_TRAIN_STEPS
    ],
    title=(
        "Notebook 24 — after "
        f"{PROPOSAL_TRAIN_STEPS} spatial/proposal micro-steps"
    ),
)


# 9. Phase C — does the new mechanism break the old source-9 symmetry?

Run the full **spatial-only** proposal/query path after Phase B, still with empty temporal state.

This is the first downstream test after proposal learning.

The most important comparison is:

```text
legacy step-30:
    same source centroid
    nearly symmetric query slots

proposal architecture:
    multiple learned anchors
    proposal-local identities
```

A large reduction in sibling-mask similarity is desirable, but after only spatial/proposal training the query decoder itself has not yet been optimized for the new query semantics. Therefore Phase C is mainly structural.


In [ ]:
phase_c_row, phase_c_artifacts = (
    evaluate_full_snapshot(
        proposal_model,
        proposal_cfg,
        query_mode="spatial_proposals",
        label=(
            "spatial_proposals_after_"
            f"{PROPOSAL_TRAIN_STEPS}_proposal_steps"
        ),
    )
)

phase_c_compare_df = pd.DataFrame(
    [
        legacy_row,
        proposal_step0_row,
        phase_c_row,
    ]
)

phase_c_compare_df.to_csv(
    RUN_DIR / "phase_c_symmetry_comparison.csv",
    index=False,
)

display(
    phase_c_compare_df[
        [
            "label",
            "source9_seed_query_count",
            "source9_recall_1p0_all_learned",
            "initial_pair_dist_mean_dref",
            "initial_pair_dist_min_dref",
            "initial_embedding_cos_mean",
            "initial_embedding_cos_max",
            "mask_pair_dice_mean",
            "mask_pair_dice_median",
            "source9_distinct_gt_matched_by_seed_type",
            "source9_matched_coarse_dice_mean",
        ]
    ]
)


# 10. Phase D — short spatial-only query bootstrap

This phase runs **only if the Phase-B proposal gate passes**.

Trainable groups:

```text
spatial   LR × 0.25
dense     LR × 0.50
proposal  LR × 1.00
query     LR × 1.00
```

Temporal and native groups remain frozen and the temporal state remains empty.

Loss semantics match the repository's query-bootstrap stage:

```text
native Dice/focal = 0
count             = 0
overlap           = 0

existence          active
coarse Dice/focal  active
center             active
dense spatial      active
internal boundary  active
proposal center    active
aux query layers   active
```


In [ ]:
def configure_query_bootstrap(model):
    groups = model_parameter_groups(model)

    for parameter in model.parameters():
        parameter.requires_grad_(False)

    lr_scales = {
        "spatial": 0.25,
        "dense": 0.50,
        "proposal": 1.00,
        "query": 1.00,
    }

    for group_name in lr_scales:
        for parameter in groups[group_name]:
            parameter.requires_grad_(True)

    optimizer = torch.optim.AdamW(
        [
            {
                "name": group_name,
                "params": groups[group_name],
                "lr": BASE_LR * scale,
            }
            for group_name, scale
            in lr_scales.items()
        ],
        lr=BASE_LR,
        weight_decay=proposal_cfg.training.weight_decay,
    )

    return optimizer


query_criterion = RefinementCriterion(
    proposal_cfg.losses,
    proposal_cfg.queries,
    proposal_cfg.training,
    proposal_cfg.proposals,
).to(device)

query_criterion.set_loss_weight_overrides(
    {
        "dice_hi": 0.0,
        "focal_hi": 0.0,
        "count": 0.0,
        "overlap": 0.0,
    }
)


def run_query_bootstrap(model):
    optimizer = configure_query_bootstrap(
        model
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=True,
        init_scale=1024.0,
    )

    metric_rows = []
    train_rows = []

    for step in range(
        QUERY_TRAIN_STEPS + 1
    ):
        if step in QUERY_EVAL_STEPS:
            print(
                f"[Phase D] evaluating step {step} ..."
            )

            row, _ = evaluate_full_snapshot(
                model,
                proposal_cfg,
                query_mode="spatial_proposals",
                label=f"query_bootstrap_step_{step}",
            )

            row["step"] = int(step)
            metric_rows.append(row)

            print(
                "  matched source9 GT by proposal queries = "
                f"{row['source9_distinct_gt_matched_by_seed_type']}/9 | "
                "mask pair Dice = "
                f"{row['mask_pair_dice_mean']:.3f}"
            )

        if step == QUERY_TRAIN_STEPS:
            break

        model.train()
        query_criterion.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        torch.cuda.reset_peak_memory_stats()
        step_started = time.perf_counter()

        with torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        ):
            outputs = (
                forward_spatial_only_full(
                    model,
                    query_mode="spatial_proposals",
                )
            )

            losses = query_criterion(
                outputs,
                targets,
            )

        scaler.scale(
            losses["loss"]
        ).backward()

        scaler.unscale_(
            optimizer
        )

        grad_norm = torch.nn.utils.clip_grad_norm_(
            [
                parameter
                for parameter in model.parameters()
                if parameter.requires_grad
            ],
            float(
                proposal_cfg.training.max_grad_norm
            ),
        )

        scaler.step(optimizer)
        scaler.update()

        train_rows.append(
            {
                "from_step": step,
                "to_step": step + 1,
                **{
                    key: float(
                        value.detach()
                        .float()
                        .cpu()
                    )
                    for key, value
                    in losses.items()
                },
                "grad_norm_before_clip": float(
                    grad_norm.detach()
                    .float()
                    .cpu()
                ),
                "peak_cuda_gib": float(
                    torch.cuda.max_memory_allocated()
                    / 1024**3
                ),
                "step_seconds": float(
                    time.perf_counter()
                    - step_started
                ),
            }
        )

        del outputs, losses
        gc.collect()
        torch.cuda.empty_cache()

    return (
        pd.DataFrame(metric_rows),
        pd.DataFrame(train_rows),
    )


In [ ]:
phase_d_ran = False
phase_d_df = pd.DataFrame()
phase_d_training_df = pd.DataFrame()

if (
    proposal_gate_passed
    and RUN_QUERY_BOOTSTRAP_IF_GATE_PASSES
):
    print(
        "Phase-B proposal gate passed. "
        "Running short spatial-only query bootstrap."
    )

    phase_d_ran = True

    phase_d_df, phase_d_training_df = (
        run_query_bootstrap(
            proposal_model
        )
    )

    phase_d_df.to_csv(
        RUN_DIR / "phase_d_query_metrics.csv",
        index=False,
    )

    phase_d_training_df.to_csv(
        RUN_DIR / "phase_d_training.csv",
        index=False,
    )

    display(
        phase_d_df[
            [
                "step",
                "source9_seed_query_count",
                "source9_distinct_gt_matched_by_seed_type",
                "source9_matched_coarse_dice_mean",
                "initial_pair_dist_mean_dref",
                "final_pair_dist_mean_dref",
                "initial_embedding_cos_mean",
                "final_embedding_cos_mean",
                "mask_pair_dice_mean",
                "mask_pair_dice_median",
                "source9_recall_1p0_all_learned",
                "foreground_hard_dice",
            ]
        ]
    )
else:
    print(
        "Phase D skipped: proposal gate did not pass "
        "or RUN_QUERY_BOOTSTRAP_IF_GATE_PASSES=False."
    )


# 11. Automatic research verdict

This verdict is deliberately a **bottleneck-localization verdict**, not a claim that the model is fully solved.

The logic is:

### Proposal gate fails
If source-9 learned proposal recall remains below 8/9 at 1 dref after the short spatial-only optimization:

```text
next target = spatial representation / proposal mechanism
```

A more aggressive spatial-backbone or high-resolution instance-geometry redesign is justified.

### Proposal gate passes but query separation remains weak
If proposals recover the cells but short query optimization still produces collapsed masks:

```text
next target = query decoder / mask basis
```

Do not keep modifying the proposal detector.

### Proposal gate passes and query masks separate
Then the original “multiple slots but one spatial identity” defect has been causally broken. Move downstream to missing-cell, false-positive, and native-mask tests.


In [ ]:
legacy_mask_dice = float(
    legacy_row["mask_pair_dice_mean"]
)

if not proposal_gate_passed:
    verdict = (
        "RED_PROPOSAL_MECHANISM_NOT_YET_RECOVERING_SOURCE9"
    )
    next_action = (
        "The new query semantics are not yet the limiting factor. "
        "Inspect/redesign the spatial proposal representation or "
        "the high-resolution spatial backbone because the model still "
        "does not create enough useful source-9 hypotheses."
    )

elif not phase_d_ran:
    verdict = (
        "GREEN_PROPOSAL_GATE_PASSED_QUERY_STAGE_NOT_RUN"
    )
    next_action = (
        "The spatial proposal mechanism recovered source 9. "
        "Run the short spatial-only query bootstrap before changing "
        "the spatial backbone again."
    )

else:
    phase_d_final = (
        phase_d_df[
            phase_d_df["step"]
            == QUERY_TRAIN_STEPS
        ]
        .iloc[0]
    )

    distinct_gt = int(
        phase_d_final[
            "source9_distinct_gt_matched_by_seed_type"
        ]
    )

    final_mask_dice = float(
        phase_d_final[
            "mask_pair_dice_mean"
        ]
    )

    mask_separation_gain = (
        legacy_mask_dice
        - final_mask_dice
    )

    if (
        distinct_gt >= 7
        and mask_separation_gain >= 0.10
    ):
        verdict = (
            "GREEN_SPATIAL_IDENTITY_BOTTLENECK_BROKEN"
        )
        next_action = (
            "The new proposal pathway creates useful source-9 identities "
            "and the query masks are separating. Next test missing-cell "
            "discovery, false-positive rejection, and then native rendering "
            "before any full staged overfit."
        )
    else:
        verdict = (
            "YELLOW_PROPOSALS_WORK_QUERY_MASK_STAGE_IS_NEXT_BOTTLENECK"
        )
        next_action = (
            "Proposal recall passed, so do not redesign the proposal "
            "detector first. Inspect the query decoder, proposal-query "
            "embedding path, attention support, and coarse mask basis."
        )

print("=" * 92)
print("NOTEBOOK 24 — SPATIAL PROPOSAL CAUSAL VERDICT")
print("=" * 92)
print("Verdict:", verdict)
print()
print("Phase-B proposal gate:", proposal_gate_passed)
print(
    "Final source9 proposal recall @1 dref:",
    float(
        phase_b_final[
            "source9_recall_1p0_all_learned"
        ]
    ),
)
print("Phase-D ran:", phase_d_ran)

if phase_d_ran:
    print(
        "Final distinct source9 GT matched by proposal queries:",
        int(
            phase_d_final[
                "source9_distinct_gt_matched_by_seed_type"
            ]
        ),
        "/ 9",
    )
    print(
        "Legacy → final proposal mask pair Dice:",
        f"{legacy_mask_dice:.3f}",
        "→",
        f"{final_mask_dice:.3f}",
    )

print()
print("Next action:")
print(next_action)


# 12. Compact final comparison and saved manifest


In [ ]:
final_rows = [
    {
        "stage": "legacy_step30",
        **legacy_row,
    },
    {
        "stage": "proposal_step30",
        **proposal_step0_row,
    },
    {
        "stage": (
            f"proposal_after_{PROPOSAL_TRAIN_STEPS}"
            "_spatial_steps"
        ),
        **phase_c_row,
    },
]

if phase_d_ran:
    final_rows.append(
        {
            "stage": (
                f"proposal_after_{QUERY_TRAIN_STEPS}"
                "_query_steps"
            ),
            **phase_d_final.to_dict(),
        }
    )

final_comparison_df = pd.DataFrame(
    final_rows
)

final_comparison_df.to_csv(
    RUN_DIR / "final_comparison.csv",
    index=False,
)

display(
    final_comparison_df[
        [
            "stage",
            "source9_recall_1p0_all_learned",
            "source9_seed_query_count",
            "source9_distinct_gt_matched_by_seed_type",
            "source9_matched_coarse_dice_mean",
            "initial_pair_dist_mean_dref",
            "initial_embedding_cos_mean",
            "mask_pair_dice_mean",
            "foreground_hard_dice",
            "source9_internal_boundary_auc",
        ]
    ]
)

report = {
    "notebook": 24,
    "checkpoint": str(
        STEP30_CHECKPOINT
    ),
    "source_id": SOURCE_ID,
    "source9_gt_ids": (
        source9_gt_ids.tolist()
    ),
    "proposal_train_steps": (
        PROPOSAL_TRAIN_STEPS
    ),
    "query_train_steps": (
        QUERY_TRAIN_STEPS
    ),
    "proposal_gate_threshold_recall_1dref": (
        PROPOSAL_GATE_RECALL_1DREF
    ),
    "proposal_gate_passed": bool(
        proposal_gate_passed
    ),
    "phase_d_ran": bool(
        phase_d_ran
    ),
    "verdict": verdict,
    "next_action": next_action,
    "phase_b_seconds": float(
        phase_b_seconds
    ),
    "files": [
        path.name
        for path in sorted(
            RUN_DIR.glob("*")
        )
    ],
}

with (
    RUN_DIR / "verdict.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        report,
        handle,
        indent=2,
        default=float,
    )

print(
    json.dumps(
        report,
        indent=2,
        default=float,
    )
)

gc.collect()
torch.cuda.empty_cache()

print()
print(
    "Notebook 24 complete. "
    "No full staged overfit was run."
)
